# Customer Churn Analysis

## Project Overview

Customer churn is one of the major challenges faced by subscription-based businesses. It refers to the situation in which customers discontinue their subscriptions or stop using a company's services.

This project focuses on analyzing customer churn patterns to understand why customers leave, which customer segments experience higher churn, and how businesses can improve customer retention.

Using Python, SQL, Pandas, NumPy, Matplotlib, and Seaborn, this project explores customer demographics, subscription details, monthly charges, and customer support interactions to identify meaningful patterns and business insights.

## Project Objectives

- Calculate the overall customer churn and retention rates.
- Identify customer segments with higher churn rates.
- Analyze churn across subscription plans, contract types, and geographical locations.
- Evaluate customer revenue, average revenue per user (ARPU), and revenue at risk.
- Examine the relationship between customer support interactions, escalations, and churn.
- Visualize important trends and relationships using statistical charts.
- Generate actionable business recommendations to improve customer retention.

## Dataset Description

The analysis uses three interconnected datasets:

1. **Customer Data:** Contains customer information such as customer ID, name, gender, date of birth, state, and country.
2. **Subscription Data:** Contains subscription type, plan type, contract details, monthly charges, renewal dates, cancellation dates, and churn-related information.
3. **Customer Support Data:** Contains complaint dates, escalation records, CSAT scores, and customer comments.

The datasets are connected using the common `customerid` field.

## Tools and Technologies

- Python – Data analysis and manipulation
- Pandas & NumPy – Data cleaning, transformation, and calculations
- SQLite – Database connectivity and data extraction
- Matplotlib & Seaborn – Data visualization
- Jupyter Notebook – Interactive analysis and documentation

## Project Methodology

The project follows a structured data analytics workflow:

1. Data extraction from the SQLite database.
2. Data inspection and understanding.
3. Data cleaning and preprocessing.
4. Feature engineering and customer-level data aggregation.
5. Dataset integration using customer IDs.
6. Exploratory Data Analysis (EDA).
7. Statistical analysis and data visualization.
8. Interpretation of findings and business recommendations.

## Expected Outcome

The objective is to transform raw customer, subscription, and support data into meaningful insights that help understand churn behavior, identify at-risk customer segments, and support data-driven retention strategies.

## 1. Importing Libraries

In this section, we import the Python libraries required for data manipulation, numerical operations, database connectivity, and data visualization.


In [1]:
# Importing required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import seaborn as sns


## 2. Database Connection & Data Loading

In this section, we establish a connection with the SQLite database and retrieve the available tables into Pandas DataFrames.

This allows us to work with customer, subscription, and support data directly in Python.


In [2]:

# Connecting to the SQLite database

connection = sqlite3.connect('customer_churn_analysis.db')

# Fetching available table names
sql_query = """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
"""

tables = pd.read_sql(sql_query, connection)

# Loading tables into separate DataFrames
for table_name in tables['name']:
    df_table = pd.read_sql(f"SELECT * FROM {table_name}", connection)
    globals()[f"df_{table_name}"] = df_table

    print(f"{table_name}: {df_table.shape}")

connection.close()


db_customer: (5000, 8)
db_subscription: (5000, 11)
db_support: (10563, 6)


## 3. Data Understanding & Initial Inspection

Before performing data cleaning and analysis, we inspect the structure of each dataset.

This includes checking the number of records, column names, data types, and sample records to understand the data and identify potential issues.

In [3]:
# Checking the structure of customer data

print("Customer Data:")
print(df_db_customer.info())

print("\nSubscription Data:")
print(df_db_subscription.info())

print("\nSupport Data:")
print(df_db_support.info())

Customer Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   customerid  5000 non-null   object 
 1   name        5000 non-null   object 
 2   country     4840 non-null   object 
 3   state       5000 non-null   object 
 4   gender      5000 non-null   object 
 5   dob         5000 non-null   object 
 6   interests   1114 non-null   object 
 7   pincode     4404 non-null   float64
dtypes: float64(1), object(7)
memory usage: 312.6+ KB
None

Subscription Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               5000 non-null   object 
 1   subscription_start_date  5000 non-null   object 
 2   subscription_type        5000 non-null   object 
 3   renewal_

## 2. Data Cleaning

### A. Customer Data Cleaning

Before cleaning the customer dataset, we inspect the first and last few records to understand the existing data.

In [4]:
# Checking first 5 records
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,NaN
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,None,NaN
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,NaN
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,None,NaN
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,NaN


In [5]:
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
4995,9986-IGJVG,sanjay,India,Punjab,Male,1991-04-30 00:00:00,None,142235.0
4996,9986-MTVDR,karan,India,Gujarat,Male,1989-04-08 00:00:00,None,372560.0
4997,9992-ELMRV,om,India,Odisha,Male,1978-04-01 00:00:00,kids,763182.0
4998,9994-ECAVK,eshan,India,Karnataka,Male,1994-05-13 00:00:00,None,572266.0
4999,9998-JTGPH,ankit,India,Haryana,Male,1983-08-17 00:00:00,None,124188.0


In [6]:
# A. RENAME COLUMN - NAME
# B. DROP COLUMNS -INTEREST AND PINCODE 
# C. CHANGE DATATYPE - DOB
# D. DATA STANDARDIZATION
# E. FIX MISSING VALUE - COUNTRY

### A. Rename Column – Name

The `name` column in the customer dataset is renamed to `customer_name` to make the column name more descriptive and consistent with the analysis.

In [7]:
# Renaming name column

df_db_customer.rename(columns={'name': 'customer_name'}, inplace=True)

df_db_customer.head()

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,NaN
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,None,NaN
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,NaN
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,None,NaN
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,NaN


### B. Drop Columns – Interests and Pincode

The `interests` and `pincode` columns are removed from the customer dataset as they are not required for the current churn analysis.

In [8]:
# Dropping unnecessary columns

df_db_customer.drop(df_db_customer.columns[-2:],axis=1, inplace = True)

df_db_customer.head()

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00


### C. Change Datatype – DOB

The `dob` column is converted from object datatype to datetime format. This allows us to perform date-based operations and calculations during analysis.

In [9]:
# Converting DOB column to datetime

df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customerid     5000 non-null   object        
 1   customer_name  5000 non-null   object        
 2   country        4840 non-null   object        
 3   state          5000 non-null   object        
 4   gender         5000 non-null   object        
 5   dob            5000 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 234.5+ KB


### D. Data Standardization

The gender column is checked for unique values to identify inconsistent naming conventions. The values `Men` and `Women` are standardized to `Male` and `Female` respectively.

In [10]:
# Checking unique gender values

df_db_customer['gender'].unique()

array(['Male', 'Female', 'Women', 'Men'], dtype=object)

In [11]:
# Standardizing gender values

df_db_customer['gender'] = df_db_customer['gender'].replace({
    'Men':'Male' , 
    'Women' : 'Female'})

# Verifying standardized values

df_db_customer['gender'].unique()

array(['Male', 'Female'], dtype=object)

### E. Fix Missing Values – Country

The country column is checked for missing values. We use the existing state and country relationships to fill missing country values wherever a reliable mapping is available.

To identify the relationship between states and countries, we create a mapping using records where the country value is already available. This mapping will be used to fill missing country values.

In [12]:
# Checking rows where country is missing

df_db_customer[df_db_customer['country'].isna()]


# Creating state-country mapping

state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

# Filling missing country values using state-country mapping

df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

# Checking remaining missing country values
df_db_customer[df_db_customer['country'].isna()][['customer_name', 'state']]


,customer_name,state
